# 1 Configuration

In [0]:
SOURCE_PATH = "s3://<bucket>/supplier-delivery-feed/"
CHECKPOINT_PATH = "/Volumes/.../checkpoints/supplier-deliveries-feed/"
SCHEMA_PATH = "/Volumes/.../schemas/supplier-deliveries-feed/"
TARGET_TABLE = "walmart.bronze.supplier_deliveries_b"

# 2 Read with Auto Loader

In [0]:
from pyspark.sql.functions import col

df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("header", "true")
         .option("cloudFiles.schemaLocation", SCHEMA_PATH)
         .option("cloudFiles.schemaEvolutionMode", "rescue")
         .load(SOURCE_PATH)
)

# 3 Write to Bronze

In [0]:
(
    df.writeStream
      .option("checkpointLocation", CHECKPOINT_PATH)
      .trigger(availableNow=True)
      .toTable(TARGET_TABLE)
)

# 4 Verification

In [0]:
%sql
SELECT COUNT(*)
FROM walmart.bronze.supplier_deliveries_b;

In [0]:
%sql
SELECT MIN(delivery_date), MAX(delivery_date) FROM walmart.bronze.supplier_deliveries_b;